In [1]:
import pandas as pd
# import cellbell
import requests
import json
import threading
import os
import time
from requests.utils import quote
from joblib import Parallel, delayed
import multiprocessing
import numpy as np
pd.options.mode.chained_assignment = None

In [2]:
global TempDF

TempDF=pd.DataFrame(columns=['Query','PII','DOI','Title', 'Journal'])
TempDF=TempDF.set_index('DOI')

In [3]:
# Ti, Zr, V, Nb, Ta

In [4]:
query = 'carbide Si'
query2 = '_'.join(query.split())
query2

'carbide_Si'

In [5]:
pwd

'/scratch16/mshiel10/mzaki4/cerie_results/corpus'

In [6]:
output_dir = f'./dois_elsevier_{query2}_'
# output_dir = f'./dois_elsevier_'

os.makedirs(output_dir, exist_ok = True)


In [11]:
jlist = ['0272-8842'] # ['1873-3956'] # journal isbn
jlist = ['0925-8388'] # journal of alloy and compounds
jlist = ['0955-2219'] # journal of europian ceramics society
jlist = ['1879-0496'] # corrosion science
jlist = ['0263-4368'] # INTERNATIONAL JOURNAL OF REFRACTORY METALS HARD MATERIALS	
jlist = ['0257-8972'] # SURFACE COATINGS
jlist = ['1005-0302','1385-8947','1359-6462'] # JOURNAL OF MATERIALS SCIENCE TECHNOLOGY	, chemical engineering journal, scripta materialia
jlist = ['1359-8368', '1359-6454','0921-5093' ,'0167-577X'] # COMPOSITES PART B ENGINEERING	, acta materialia, MATERIALS SCIENCE AND ENGINEERING A STRUCTURAL MATERIALS PROPERTIES MICROSTRUCTURE AND PROCESSING	, MATERIALS SCIENCE LETTERS
jlist = ['1875-5372','0254-0584'] # RARE METAL MATERIALS AND ENGINEERING	, MATERIALS CHEMISTRY AND PHYSICS
jlist = ['0010-938X'] # corrosion science
jlist = ['0042-207X'] # vacuum
jlist = ['0026-0657'] # metal powder report
jlist = ['0022-3115'] # journal of nuclear materials
jlist = ['2468-0230'] # surfaces and interfaces
jlist = ['0927-0256'] # computational materials science
jlist = ['0043-1648'] # wear
jlist = ['1879-0062'] # diamond and related materials
jlist = ['1873-1961'] # engineering failure analysis
jlist = ['0301-679X'] # tribology international
jlist = ['1873-5002'] # journal of crystal growth
jlist = ['1873-3891'] # carbon

In [12]:
jlist = ['0272-8842',
'0925-8388',
'0955-2219',
'1879-0496',
'0263-4368',
'0257-8972',
'1005-0302','1385-8947','1359-6462',
'1359-8368','1359-6454','0921-5093' ,'0167-577X',
'1875-5372','0254-0584',
'0010-938X',
'0042-207X',
'0026-0657',
'0022-3115',
'2468-0230',
        '0927-0256',
         '0043-1648','1879-0062','1873-1961','0301-679X','1873-5002','1873-3891']

In [13]:
email = 'mzaki4@jh.edu'

In [14]:
# jlist=  ['2468-0230','0927-0256']

In [ ]:
for journal_issn in jlist:#[jlist[0]]:
    for dates  in [('1600-01','2005-01'),('2005-01','2010-01'),('2010-01','2015-01'),('2015-01','2020-01'),('2020-01','2024-01'),('2024-01','2026-01')]:
        global TempDF

        TempDF=pd.DataFrame(columns=['Query','PII','DOI','Title', 'Journal'])
        TempDF=TempDF.set_index('DOI')
        
        start_date = dates[0]
        end_date =  dates[1]
        def cross_reference_search(query,TempDF):

            cursor = "*"
            keep_paging = True
            max_rows = 1000


            base_url = 'https://api.crossref.org/works?query='

            headers = {
                   'Accept': 'application/json',
                   'mailto':   email
                 }

            # params = {
            # 'filter': f'from-pub-date:{start_date},container-title:{journal_name}',
            #     }
            params = {
            'filter': f'from-pub-date:{start_date},until-pub-date:{end_date},issn:{journal_issn}',
                }
            while (keep_paging):

                try:
                    # https://api.crossref.org/works?query=+&filter=from-pub-date%3A2019-01%2Ccontainer-title%3ACeramics+International&rows=1000&cursor=%2A
                    print(query)
                    r = requests.get(base_url + query + "&rows=" + str(max_rows) + "&cursor=" + cursor,
                                              headers=headers, timeout=100,params=params)#, proxies=proxies)

                    cursor = quote(r.json()['message']['next-cursor'], safe='')

                    # print(r.url)

                    if len(r.json()['message']['items']) == 0:
                                    keep_paging = False

                    for item in r.json()['message']['items']:
                        try:
                            Journal=item['container-title'][0]
                        except:
                            print('here')
                            Journal='None'

                        TempDF.loc[item['DOI']]=(query,'None',item['title'][0],Journal)
                        if TempDF.shape[0]%500 == 0:
                            time.sleep(1)
                            # pass
                            # print('Papers found = %d'%(int(TempDF.shape[0])))

                        # print('ek mila')


                except Exception as e:
                    # print(e)
                    keep_paging = False

            return None

        # query = 'high temperature ceramics'
        # query = 'high entropy ceramics'
        exact_query = f'"{query}"'
        cross_reference_search(exact_query,TempDF)
        s1 = start_date.split('-')[0]
        s2 = end_date.split('-')[0]
        try:
            journal = list(TempDF['Journal'])[0].strip()
            jname = journal.replace(' ','_')
            output_dir = os.path.join(f'./dois_elsevier_{query2}_/{jname}')
            os.makedirs(output_dir, exist_ok = True)
            TempDF.to_csv(os.path.join(output_dir, f'{journal_issn}_{s1}_{s2}.csv'))
            print(len(TempDF), 'dois found for',journal_issn,'from',s1,'to',s2)
        except IndexError:
            print('none for',journal_issn, 'from', s1, 'to', s2)
        time.sleep(5)
    # cellbell.ding()

"carbide Si"
"carbide Si"
86 dois found for 0272-8842 from 1600 to 2005
"carbide Si"
"carbide Si"
53 dois found for 0272-8842 from 2005 to 2010
"carbide Si"
"carbide Si"
178 dois found for 0272-8842 from 2010 to 2015
"carbide Si"
"carbide Si"
530 dois found for 0272-8842 from 2015 to 2020
"carbide Si"
"carbide Si"
549 dois found for 0272-8842 from 2020 to 2024
"carbide Si"
"carbide Si"
313 dois found for 0272-8842 from 2024 to 2026
"carbide Si"
"carbide Si"
279 dois found for 0925-8388 from 1600 to 2005
"carbide Si"
"carbide Si"
305 dois found for 0925-8388 from 2005 to 2010
"carbide Si"
"carbide Si"
428 dois found for 0925-8388 from 2010 to 2015
"carbide Si"
"carbide Si"
763 dois found for 0925-8388 from 2015 to 2020
"carbide Si"
"carbide Si"
644 dois found for 0925-8388 from 2020 to 2024


In [7]:
# jlist = os.listdir(f'./dois_elsevier_{query2}_/')

jlist = os.listdir(f'./dois_elsevier_/')

In [8]:
keys = ['b48a47cd4290ccc9a05980f26232e186','008ea37d47c10f13ddbd130c659bd605']


In [9]:
import os

In [10]:
table_dir = '/scratch16/mshiel10/mzaki4/cerie/'
cache_dir = os.path.join(table_dir, '.cache')


In [ ]:
masterdf=pd.DataFrame(columns=['DOI', 'Query', 'PII', 'Title', 'Journal'])
masterdf = masterdf[0:0] 
# masterdf=TempDF.set_index('PII')

for ele in ['Si', 'Ti', 'Zr', 'Nb', 'V', 'Ta','Hf']:
    journals = os.listdir(f'./dois_elsevier_carbide_{ele}_')
    for journal in journals:
        for file in os.listdir(f'./dois_elsevier_carbide_{ele}_/{journal}'):
            try:
                tempdf = pd.read_csv(f'./dois_elsevier_carbide_{ele}_/{journal}/{file}')
                masterdf = pd.concat([masterdf, tempdf])
            except:
                print(file)


In [14]:
masterdf.shape

(61231, 5)

In [15]:
masterdf = masterdf.drop_duplicates(subset='DOI', keep='first')

In [16]:
df2 = pd.read_csv('/home/mzaki4/scr16_mshiel10/mzaki4/mzaki4/cerie_results/corpus/downloaded_r1.csv')
df2.shape

(186655, 6)

In [17]:
len(set(df2.doi).intersection(set(masterdf.DOI)))

15691

In [21]:
done = set(df2.doi).intersection(set(masterdf.DOI))

In [22]:
mask = masterdf.DOI.apply(lambda x: x not in done)

In [28]:
masterdf[mask].sample(sum(mask))

,DOI,Query,PII,Title,Journal
160,10.1016/j.msea.2011.03.018,"""carbide Ti""",None,Thermal stability of in situ synthesized (TiB+...,Materials Science and Engineering: A
102,10.1016/s0167-577x(03)00455-5,"""carbide Zr""",None,Ba/Zr=1:1 freeze-dried and conventional chlori...,Materials Letters
82,10.1016/j.surfcoat.2023.129365,"""carbide Zr""",None,Enhanced corrosion and wear resistance via the...,Surface and Coatings Technology
7,10.1016/j.vacuum.2023.112009,"""carbide Zr""",None,Creep behavior and creep mechanism of Mg-Gd-Y-...,Vacuum
37,10.1016/j.corsci.2024.112520,"""carbide Ti""",None,Design of new β-type Ti alloys with outstandin...,Corrosion Science
...,...,...,...,...,...
202,10.1016/j.msea.2024.146922,"""carbide Ti""",None,Computationally guided alloy design and micros...,Materials Science and Engineering: A
30,10.1016/j.compositesb.2023.110696,"""carbide Ti""",None,Enhancing microstructure refinement and streng...,Composites Part B: Engineering
19,10.1016/j.vacuum.2013.12.007,"""carbide Ti""",None,Performance characterization on a novel interm...,Vacuum
2,10.1016/j.jmst.2023.09.008,"""carbide Nb""",None,Simultaneous enhancement of strength and ducti...,Journal of Materials Science &amp; Technology


In [24]:
import requests
import os
import pickle

import pandas  as  pd
from bs4 import BeautifulSoup
from tqdm import tqdm
import random
import threading

table_dir = './corpus'

def article_downloader(idx):
    '''
    Function to download XMLs from DOIs of Research Papers published in Elsevier journals.
            Parameters:
                    doi_list (list): List of doi strings
                    output_dir (str): Path of directory where papers will be downloaded
            Returns:
                    Downloads the research paper XML in a folder named with article pii inside output_dir
    '''
    doi = masterdf.DOI.iloc[idx]
    jdir = '_'.join(masterdf.Journal.iloc[idx].split())
    output_dir = f'./corpus/{jdir}'
    os.makedirs(output_dir, exist_ok=True)
    api_id = random.randint(0,1)
    
    # for doi in tqdm(doi_list):
    xml_url = f'https://api.elsevier.com/content/article/doi/{doi}?APIKey={keys[api_id]}'
    headers = {'User-Agent': 'Mozilla/5.0'}
    url_get = requests.get(xml_url, headers=headers)
    try:
        soup = BeautifulSoup(url_get.content, 'lxml')
        pii = soup.find('xocs:pii-unformatted').text
        soup_path = os.path.join(output_dir, str(pii))
        xmlpath = os.path.join(soup_path, str(pii) + '.xml')
        os.makedirs(soup_path, exist_ok=True)
        with open(xmlpath, 'w', encoding='utf-8') as file:
            file.write(str(soup))
    except:
        with open('err.txt','a') as rep:
            msg = f'{doi}\n'
            rep.write(doi)
        rep.close()


In [25]:
masterdf.Journal = masterdf.Journal.apply(lambda x: x.replace(" &amp; ",' and '))

In [26]:
masterdf = masterdf.reset_index(drop=True)

In [28]:
email = 'mzaki4@jh.edu'
keys = ['b48a47cd4290ccc9a05980f26232e186','008ea37d47c10f13ddbd130c659bd605']


In [29]:
idx = 0
doi = masterdf.DOI.iloc[idx]
jdir = '_'.join(masterdf.Journal.iloc[idx].split())
output_dir = f'./corpus/{jdir}'
os.makedirs(output_dir, exist_ok=True)
api_id = random.randint(0,1)
headers = {'User-Agent': 'Mozilla/5.0'}

xml_url = f'https://api.elsevier.com/content/article/doi/{doi}?APIKey={keys[api_id]}'

url_get = requests.get(xml_url, headers=headers)

if url_get.status_code == 200:
    soup = BeautifulSoup(url_get.content, 'lxml')

In [31]:
num_cores = multiprocessing.cpu_count()
print(num_cores)

48


In [32]:
inputs = tqdm(np.arange(0,len(masterdf)))

processed_list = Parallel(n_jobs=40)(delayed(article_downloader)(i) for i in inputs)

100%|██████████| 61231/61231 [57:29<00:00, 17.75it/s]  


In [16]:
import requests
import os
import pickle

import pandas  as  pd
from bs4 import BeautifulSoup
from tqdm import tqdm
import random
import threading

table_dir = './corpus'

def article_downloader(idx):
    '''
    Function to download XMLs from DOIs of Research Papers published in Elsevier journals.
            Parameters:
                    doi_list (list): List of doi strings
                    output_dir (str): Path of directory where papers will be downloaded
            Returns:
                    Downloads the research paper XML in a folder named with article pii inside output_dir
    '''
    doi = masterdf.DOI.iloc[idx]
    jdir = '_'.join(masterdf.Journal.iloc[idx].split())
    output_dir = f'./corpus/{jdir}'
    os.makedirs(output_dir, exist_ok=True)
    api_id = random.randint(0,1)
    
    # for doi in tqdm(doi_list):
    xml_url = f'https://api.elsevier.com/content/article/doi/{doi}?APIKey={keys[api_id]}'
    headers = {'User-Agent': 'Mozilla/5.0'}
    url_get = requests.get(xml_url, headers=headers)
    try:
        soup = BeautifulSoup(url_get.content, 'lxml')
        pii = soup.find('xocs:pii-unformatted').text
        soup_path = os.path.join(output_dir, str(pii))
        xmlpath = os.path.join(soup_path, str(pii) + '.xml')
        os.makedirs(soup_path, exist_ok=True)
        with open(xmlpath, 'w', encoding='utf-8') as file:
            file.write(str(soup))
    except:
        with open('err.txt','a') as rep:
            msg = f'{doi}\n'
            rep.write(doi)
        rep.close()


In [36]:
masterdf.shape

(50890, 5)

In [16]:
idx = 0
doi = masterdf.DOI.iloc[idx]
jdir = '_'.join(masterdf.Journal.iloc[idx].split())
output_dir = f'./corpus/{jdir}'
os.makedirs(output_dir, exist_ok=True)
api_id = random.randint(0,1)
headers = {'User-Agent': 'Mozilla/5.0'}

xml_url = f'https://api.elsevier.com/content/article/doi/{doi}?APIKey={keys[api_id]}'

url_get = requests.get(xml_url, headers=headers)

if url_get.status_code == 200:
    soup = BeautifulSoup(url_get.content, 'lxml')

In [17]:
num_cores = multiprocessing.cpu_count()
print(num_cores)

48


In [ ]:
inputs = tqdm(np.arange(0,len(masterdf)))

processed_list = Parallel(n_jobs=num_cores)(delayed(article_downloader)(i) for i in inputs)

  0%|          | 96/101767 [00:20<2:49:05, 10.02it/s]